In [1]:
# ===============================
# Animal Dataset Cleaning Project
# ===============================

import pandas as pd
import numpy as np

# 1️⃣ Load CSV as strings to inspect raw data
# I want to see exactly what’s in the file, including messy stuff
df = pd.read_csv("animal_data_dirty1.csv", sep=';', dtype=str)

# Preview first 10 rows
print("First 10 rows of raw data:")
print(df.head(10))

# Show original column names
print("\nOriginal column names:")
print(df.columns.tolist())

# 2️⃣ Standardize column names (fix typos, remove spaces)
df.columns = df.columns.str.strip()  # remove extra spaces
df.rename(columns={
    'Animal type': 'Animal_Type',
    'Weight kg': 'Weight_kg',
    'Body Length cm': 'Body_Length_cm',
    'Animal code': 'Animal_Code',
    'Animal name': 'Animal_name',
    'Observation date': 'Observation_Date',
    'Data compiled by': 'Data_Completed_by'
}, inplace=True)

print("\nRenamed columns:")
print(df.columns.tolist())

# 3️⃣ Replace string 'NaN' and empty strings with actual np.nan
df.replace('NaN', np.nan, inplace=True)
for col in df.select_dtypes(include='string').columns:
    df[col] = df[col].str.strip()       # remove leading/trailing spaces
    df[col] = df[col].replace('', np.nan)

# 4️⃣ Check missing values per column
print("\nMissing values per column:")
print(df.isna().sum())

# 5️⃣ Notice: Animal_Code has 1011 missing values
# This column is completely empty, so there’s nothing to fill or keep
# Decision: Drop this column
if df['Animal_Code'].isna().all():
    df.drop(columns=['Animal_Code'], inplace=True)
    print("\nDropped 'Animal_Code' column because it is completely empty.")

# 6️⃣ Fill categorical columns safely
categorical_cols = ['Gender', 'Animal_Type', 'Animal_name', 'Country']
for col in categorical_cols:
    if df[col].mode().empty:
        df[col] = df[col].fillna('Unknown')  # fallback if mode is empty
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

# 7️⃣ Convert numeric columns to numbers and fill missing values with mean
numeric_cols = ['Weight_kg', 'Body_Length_cm', 'Latitude', 'Longitude']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')  # convert invalid strings to NaN
    df[col] = df[col].fillna(df[col].mean())           # fill missing with mean

# 8️⃣ Convert Observation_Date to datetime and fill missing with fixed placeholder
df['Observation_Date'] = pd.to_datetime(df['Observation_Date'], errors='coerce', dayfirst=True)
df['Observation_Date'] = df['Observation_Date'].fillna(pd.Timestamp('2024-01-01'))

# 9️⃣ Verify no missing values left
print("\nMissing values per column after cleaning:")
print(df.isna().sum())

# 🔟 Preview cleaned DataFrame
print("\nPreview of cleaned data:")
print(df.head())

First 10 rows of raw data:
       Animal type  Country Weight kg Body Length cm          Gender  \
0              NaN      NaN       NaN            NaN             NaN   
1              NaN      NaN       NaN            NaN             NaN   
2   European bison   Poland       930            335            male   
3   European bison   Poland       909            311  not determined   
4  European bison™   Poland       581            277          female   
5  European bisson   Poland       900            295            male   
6  European buster   Poland       620            250          female   
7             lynx  Hungary        24             76            male   
8            lynx?  Hungary        23             82          female   
9      red squirel   Poland     0.308             22          female   

  Animal code   Latitude  Longitude Animal name Observation date  \
0         NaN        NaN        NaN         NaN       03.01.2024   
1         NaN        NaN        NaN         

In [2]:
df['Gender'].mode()[0]      
df['Weight_kg'].mean()       
df['Body_Length_cm'].median() 

np.float64(21.0)